<div align=center>
  <h1>
    Gaussian Inference on Google Colab
  </h1>
  <p>
    <a href=https://mhsung.github.io/kaist-cs479-spring-2025/ target="_blank"><b>KAIST CS479: Machine Learning for 3D Data</b></a><br>
    Programming Assignment 3
  </p>
</div>

<div align=center>
  <p>
    Instructor: <a href=https://mhsung.github.io target="_blank"><b>Minhyuk Sung</b></a> (mhsung [at] kaist.ac.kr)<br>
    TA: <a href=https://choidaedae.github.io target="_blank"><b>Daehyeon Choi</b></a>  (daehyeonchoi [at] kaist.ac.kr)<br>      
    Credit: <a href=https://dvelopery0115.github.io target="_blank"><b>Seungwoo Yoo</b></a>  (dvelopery0115 [at] kaist.ac.kr)      
  </p>
</div>



This notebook guides you through the entire 3DGS assignment pipeline using a pre-trained checkpoint.

**Steps:**

1. Environment Setup (GPU check, repository clone, dependency installation)

2. Implementation (Task 1, 2, 3— edit source files as described in the README)

3. Rendering & Evaluation (Task 4)

4. Submission Packaging

## 1. Environment Setup

Before getting started, go to Runtime → Change runtime type on the top menu bar, and choose T4 GPU under Hardware Accelerator.

You should then see True and the GPU status with the commands below:


### 1.1 GPU Check

Verify that GPU runtime is enabled. Go to "`Runtime` → `Change runtime type` → `T4 GPU`."

In [ ]:
import torch
print(torch.cuda.is_available())
!nvidia-smi

True
Fri Feb 13 05:20:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+

### 1.2 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



### 1.3 Install dependencies


In [ ]:
%cd /content/drive/MyDrive/KAISTVisualAIGroup/TA/CS479-Assignment-3DGS

/content/drive/MyDrive/KAISTVisualAIGroup/TA/CS479-Assignment-3DGS


In [ ]:
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip install torchmetrics[image]
!pip install imageio[ffmpeg]
!pip install plyfile tyro jaxtyping==0.2.36 typeguard==2.13.3

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 81.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 114.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 104.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/20

In [ ]:
%cd /content/drive/MyDrive/KAISTVisualAIGroup/TA/CS479-Assignment-3DGS/simple-knn

!pip install . --no-build-isolation --verbose

/content/drive/MyDrive/KAISTVisualAIGroup/TA/CS479-Assignment-3DGS/simple-knn
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Processing /content/drive/MyDrive/KAISTVisualAIGroup/TA/CS479-Assignment-3DGS/simple-knn
  Running command python setup.py egg_info
  running egg_info
  creating /tmp/pip-pip-egg-info-_uyynank/simple_knn.egg-info
  writing /tmp/pip-pip-egg-info-_uyynank/simple_knn.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-pip-egg-info-_uyynank/simple_knn.egg-info/dependency_links.txt
  writing top-level names to /tmp/pip-pip-egg-info-_uyynank/simple_knn.egg-info/top_level.txt
  writing manifest file '/tmp/pip-pip-egg-info-_uyynank/simple_knn.egg-info/SOURCES.txt'
  /usr/local/lib/python3.12/dist-packages/torch/utils/cpp_extension.py:497: UserWarning: Attempted to use ninja as the BuildExtension backend but we could not find ninja.. Falling back to using the slow distutils backend.
    warnings.warn(msg.format('we could not find ninj

### Once installed all requirements, please restart your session.

### 2) Set `PYTHONPATH`

Register the project root so Python can find local modules (e.g., `src/...`).

In [ ]:
import os
import sys

# Make sure local modules (src/...) are discoverable
os.environ["PYTHONPATH"] = "/content/drive/MyDrive/KAISTVisualAIGroup/TA/CS479-Assignment-3DGS"
print("Python:", sys.version.split()[0])
print("PYTHONPATH:", os.environ.get("PYTHONPATH"))

## Tasks

This section mirrors the assignment tasks from `README.md`, with an explicit **Reminder** block after each task.

### Task 0. Download Data

Download the scene files (`data.zip`) from the provided Google Drive link in `README.md` and extract them into the **project root**.

After extraction, the `data` directory should look like:

```text
data
│
├── nerf_synthetic      <- Camera parameters + reference images
│   ├── chair
│   ├── drums
│   ├── lego
│   └── materials
├── chair.ply           <- Gaussian splats for "Chair" scene
├── drums.ply           <- Gaussian splats for "Drums" scene
├── lego.ply            <- Gaussian splats for "Lego" scene
└── materials.ply       <- Gaussian splats for "Materials" scene
```

> **Reminder**
> - Make sure `data/nerf_synthetic/{chair,lego,materials,drums}` exists.
> - Make sure `data/{chair,lego,materials,drums}.ply` exists.

In [ ]:
%cd /content/drive/MyDrive/KAISTVisualAIGroup/TA/CS479-Assignment-3DGS/

In [ ]:
from pathlib import Path

expected_scenes = ["chair", "lego", "materials", "drums"]

missing = []
for s in expected_scenes:
    if not Path(f"data/{s}.ply").exists():
        missing.append(f"data/{s}.ply")
    if not Path(f"data/nerf_synthetic/{s}").exists():
        missing.append(f"data/nerf_synthetic/{s}")

if missing:
    print("Missing expected data paths:")
    for p in missing:
        print(" -", p)
else:
    print("Data looks OK: found .ply files and nerf_synthetic scene folders.")

### Task 1. World to NDC

Implement the coordinate transformation from world space to normalized device coordinates (NDC) in the `project_ndc` method of `renderer.py`.  

Given a homogeneous coordinate $\mathbf{p}$, performing the following matrix multiplication yields the coordinate of the point in the view space:

$$
\mathbf{p}_{\text{view}} = \mathbf{p} \mathbf{V}
$$

where $\mathbf{V}$ is the view matrix (world-to-camera transformation).

To project the point onto the image plane, first perform the matrix multiplication

$$
\mathbf{p}_{\text{proj}} = \mathbf{p}_{\text{view}} \mathbf{P}
$$

where $\mathbf{P}$ is the projection matrix (camera-to-clip transformation).
Then, divide the first three components of $\mathbf{p}_{\text{ndc}}$ by the fourth component to obtain the normalized device coordinates:

$$
\tilde{\mathbf{p}} = \frac{\mathbf{p}_{\text{proj}}}{\mathbf{p}_{\text{proj}, w}}
$$

where $\mathbf{p}_{\text{proj}, w}$ is the fourth component of $\mathbf{p}_{\text{proj}}$.


Lastly, compute the binary mask indicating the points that are behind the near plane by checking whether the $z$-coordinate of $\mathbf{p}_{\text{view}}$ is greater than $z_{\text{near}}$.


### Task 2. Covariance Matrix Projection

Implement the projection of the covariance matrix onto the image plane in the `compute_cov_2d` method of `src/renderer.py`.

Constraints (from the README):
- You are only allowed to modify the code inside the block marked with `TODO` in the method.

High-level steps (from the README):
- After transforming the centers of 3D Gaussians to camera space, compute the Jacobian

$$
\mathbf{J} = \begin{bmatrix}
  \frac{f_x}{t_z} &      0          & -\frac{f_x t_x}{t_z^2} \\
  0               & \frac{f_y}{t_z} & -\frac{f_y t_y}{t_z^2} \\
  0               &      0          & 0
\end{bmatrix}
$$

- Then project the 3D covariance to 2D via:

$$
\boldsymbol{\Sigma}_{\text{2D}} = \mathbf{J} \mathbf{W} \boldsymbol{\Sigma}_{\text{3D}} \mathbf{W}^T \mathbf{J}^T,
$$

where $\mathbf{W}$ is the rigid transform component.

> **Reminder**
> - Please implement **`compute_cov_2d`** in `src/renderer.py` (only inside the `TODO` block).


### Task 3. Rendering Equation of Point-Based Radiance Fields
Implement the rendering equation for point-based radiance fields in the `render` method of `src/renderer.py`.
This method computes pixel colors by blending the colors of 2D Gaussian splats stacked on the image plane.

#### Notes (from the README)
* We assume the 2D Gaussian centers and covariance matrices are already in **image space**.
* For memory reasons, the renderer processes the image **tile-by-tile**.
* The skeleton code already computes which splats are relevant for the current tile via `in_mask`.

#### Implement the following steps at the locations marked as `TODO`:

1. **Sort the Gaussians in ascending order by depth.**

2. **Compute displacement vectors**
   Compute displacement vectors
   $$\mathbf{d}_{i,j} \in \mathbb{R}^2$$
   from pixel centers to Gaussian centers.
3. **Compute Gaussian weights**
   $$
   \mathbf{w}*{i,j} = \exp\left(-\frac{1}{2} \mathbf{d}*{i,j}^T \Sigma_j^{-1} \mathbf{d}_{i,j}\right).
   $$
4. **Perform alpha blending**
   $$
   \mathbf{C}_i = \sum_j \mathbf{c}_j \tilde{\alpha}*j \prod*{k < j} (1 - \tilde{\alpha}_k),
   $$
   where
   $$\tilde{\alpha}*j = \mathbf{w}*{i,j} \alpha_j.$$

> **Reminder**
- Please implement the `TODO`s inside the **`render`** method in `src/renderer.py`.


### Task 3. Qualitative & Quantitative Evaluation

After you complete the implementation tasks above, render all required scenes and compute metrics.

1) Render all scenes (`chair`, `lego`, `materials`, `drums`):

In [ ]:
# Render all scenes (this may take a while)
# Note: render_all.sh internally runs: CUDA_VISIBLE_DEVICES=0 python render.py --scene-type <scene>
!bash render_all.sh

Rendering scene: chair
Saving outputs to: outputs/chair
Using device: cuda
Loading Scene: chair
Loaded Scene.
Loaded Camera Data.
Initialized Renderer.
200it [03:06,  1.07it/s]
Rendering scene: lego
Saving outputs to: outputs/lego
Using device: cuda
Loading Scene: lego
Loaded Scene.
Loaded Camera Data.
Initialized Renderer.
200it [03:02,  1.10it/s]
Rendering scene: materials
Saving outputs to: outputs/materials
Using device: cuda
Loading Scene: materials
Loaded Scene.
Loaded Camera Data.
Initialized Renderer.
28it [00:20,  1.33it/s]


2) Evaluate and write `evaluation.txt`:


In [ ]:
# Evaluate rendered results and generate metrics.csv

!python evaluate.py

In [ ]:
import os, glob, zipfile

STUDENT_ID = input("Enter your student ID: ").strip()
prefix = f"{STUDENT_ID}"

# Read metrics saved by evaluate.py
assert os.path.exists("evaluation.txt"), "evaluation.txt not found. Run the evaluation cell first."
metrics_str = open("evaluation.txt").read().strip()

# Find latest test renderings
test_view_dirs = sorted(glob.glob('output/**/*'))
assert test_view_dirs, "No test renderings found. Please run rendering first."
rendered_dir = test_view_dirs[-1]

# Build ZIP
zip_name = f"{prefix}.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    # 1) source code
    for root, dirs, files in os.walk('torch_nerf'):
        dirs[:] = [d for d in dirs if d != '__pycache__']
        for f in files:
            fpath = os.path.join(root, f)
            zf.write(fpath, os.path.join(prefix, fpath))

    # 2) Test view renderings
    for png in sorted(glob.glob(f'{rendered_dir}/*.png')):
        zf.write(png, os.path.join(prefix, f"{prefix}_renderings", os.path.basename(png)))

    # 3) Metrics text file
    zf.writestr(os.path.join(prefix, f"{prefix}.txt"), metrics_str)

print(f"Submission file created: {zip_name}")
print(f"  Total files: {len(zipfile.ZipFile(zip_name).namelist())}")

> **Reminder**
> - The grading script expects `{YOUR_STUDENT_ID}.txt` in the project root.
> - Make sure `outputs/{scene}` exists for all four scenes before running evaluation.

For reference, our implementation produces the following metrics:
| Scene     | LPIPS (↓)    | PSNR (↑)    | SSIM (↑)    |
|-----------|--------------|-------------|-------------|
| Chair     |    0.037     |    27.043   |    0.953    |
| Lego      |    0.047     |    25.723   |    0.939    |
| Materials |    0.043     |    25.014   |    0.937    |
| Drums     |    0.079     |    21.583   |    0.896    |
| Average   |    0.052     |    24.841   |    0.931    |


For details on grading, refer to section [Grading](#grading) of README.md.